In [1]:
# Install
!pip install torch -q

In [2]:
# Import
import torch
import torch.nn as nn
import torch.nn.functional as F
import time


In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [4]:
# Simple attention
class Attention(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.q = nn.Linear(d_model, d_model)
        self.k = nn.Linear(d_model, d_model)
        self.v = nn.Linear(d_model, d_model)

    def forward(self, x, past_k=None, past_v=None):
        q = self.q(x)
        k = self.k(x)
        v = self.v(x)

        if past_k is not None:
            k = torch.cat([past_k, k], dim=1)
            v = torch.cat([past_v, v], dim=1)

        attn = (q @ k.transpose(-2, -1)) / (x.size(-1) ** 0.5)

        mask = torch.tril(torch.ones(attn.shape[-2:], device=x.device))
        attn = attn.masked_fill(mask == 0, float('-inf'))

        attn = F.softmax(attn, dim=-1)
        out = attn @ v

        return out, k, v


In [5]:
# Tiny model
class TinyGPT(nn.Module):
    def __init__(self, vocab_size, d_model=64):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, d_model)
        self.attn = Attention(d_model)
        self.fc = nn.Linear(d_model, vocab_size)

    def forward(self, x, past_k=None, past_v=None):
        x = self.emb(x)
        out, k, v = self.attn(x, past_k, past_v)
        logits = self.fc(out)
        return logits, k, v

In [6]:
# Tokenizer
text = "hello world"
chars = sorted(set(text))
stoi = {c:i for i,c in enumerate(chars)}
itos = {i:c for c,i in stoi.items()}

def encode(s):
    return torch.tensor([stoi[c] for c in s]).unsqueeze(0).to(device)

def decode(x):
    return "".join([itos[int(i)] for i in x])

In [7]:
# Initialize model
model = TinyGPT(len(stoi)).to(device)
model.eval()


TinyGPT(
  (emb): Embedding(8, 64)
  (attn): Attention(
    (q): Linear(in_features=64, out_features=64, bias=True)
    (k): Linear(in_features=64, out_features=64, bias=True)
    (v): Linear(in_features=64, out_features=64, bias=True)
  )
  (fc): Linear(in_features=64, out_features=8, bias=True)
)

In [8]:
# Generate without kv cache
@torch.no_grad()
def generate_no_cache(model, prompt, steps=100):
    idx = encode(prompt)

    start = time.time()

    for _ in range(steps):
        logits, _, _ = model(idx)  # recompute everything
        probs = F.softmax(logits[:, -1, :], dim=-1)
        next_token = torch.multinomial(probs, 1)
        idx = torch.cat([idx, next_token], dim=1)

    end = time.time()
    return end - start

In [9]:
# Generate with kv cache
@torch.no_grad()
def generate_kv_cache(model, prompt, steps=100):
    idx = encode(prompt)
    past_k, past_v = None, None

    start = time.time()

    for _ in range(steps):
        logits, past_k, past_v = model(idx[:, -1:], past_k, past_v)
        probs = F.softmax(logits[:, -1, :], dim=-1)
        next_token = torch.multinomial(probs, 1)
        idx = torch.cat([idx, next_token], dim=1)

    end = time.time()
    return end - start

In [10]:
# Run benchmark
t1 = generate_no_cache(model, "h", steps=200)
t2 = generate_kv_cache(model, "h", steps=200)

print("WITHOUT KV Cache Time:", t1)
print("WITH KV Cache Time:   ", t2)
print("Speedup:", t1 / t2)

WITHOUT KV Cache Time: 0.3507697582244873
WITH KV Cache Time:    0.07856249809265137
Speedup: 4.464849855090057
